In [ ]:
!pip install sacrebleu
!pip install rouge_score
!pip install vllm
!pip install trl
!pip -q install groq

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import sacrebleu
import torch
import random
import json
import time
from rouge_score import rouge_scorer
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer
from IPython.display import display, HTML
from groq import Groq

# 1. EDA

In [ ]:
train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

In [ ]:
print(train.columns)
print(train.info())
for name, df in [
    ("train", train),
    ("validation", validation),
    ("test", test)
]:
    print(
        name,
        "shape =", df.shape,
        "missing =", df.isna().sum().sum(),
        "duplicates =", df.duplicated().sum()
    )

Пропусков, дубликатов нет

In [ ]:
print("DIALOGUE:")
print(train.iloc[0]["dialogue"])
print("\nSUMMARY:")
print(train.iloc[0]["summary"])
print("\nTOPIC:")
print(train.iloc[0]["topic"])

In [ ]:
train["dialogue_words"] = train["dialogue"].str.split().str.len()
train["summary_words"] = train["summary"].str.split().str.len()
train[["dialogue_words", "summary_words"]].describe()

,dialogue_words,summary_words
count,12460.000000,12460.000000
mean,130.986998,22.867657
std,70.976634,10.705946
min,40.000000,5.000000
25%,85.000000,15.000000
50%,116.000000,21.000000
75%,163.000000,28.000000
max,985.000000,153.000000


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train["dialogue_words"], bins=50)
axes[0].set_title("Распределение длины диалогов")
axes[0].set_xlabel("Количество слов")
axes[0].set_ylabel("Количество диалогов")
axes[0].grid(alpha=0.3)

axes[1].hist(train["summary_words"], bins=30)
axes[1].set_title("Распределение длины саммари")
axes[1].set_xlabel("Количество слов")
axes[1].set_ylabel("Количество саммари")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

Большинство диалогов содержит от 85 до 163 слов, однако присутствуют длинные выбросы до 985 слов. Саммари значительно короче исходных диалогов: в среднем около 23 слов. Таким образом, задача предполагает существенное сжатие исходного текста, а распределение длин имеет выраженный длинный хвост, что необходимо учитывать при выборе длины контекста.

# 2. Бейзлайн без ML

In [ ]:
nltk.download("punkt")
nltk.download("punkt_tab")

def first_last_sentences(dialogue):
    dialogue = re.sub(r'#Person\d+#:\s*', '', dialogue)
    sentences = sent_tokenize(dialogue)

    if len(sentences) == 0:
        return ""

    if len(sentences) == 1:
        return sentences[0]

    return sentences[0] + " " + sentences[-1]


test["prediction"] = test["dialogue"].apply(first_last_sentences)


test.to_csv(
    "/content/baseline_predictions.csv",
    index=False
)

# 3. LLM без Fine-Tune

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)


def make_prompt(dialogue):
    return f"""Summarize the following dialogue.
Write one concise sentence describing the main event or action.
Do not add information that is not stated in the dialogue.
Output only the summary.

Dialogue:
{dialogue}

Summary:"""


def generate_summary(dialogue):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": make_prompt(dialogue)
            }
        ],
        temperature=0,
        max_tokens=128
    )

    return response.choices[0].message.content.strip()

predictions = []

for dialogue in tqdm(test["dialogue"], desc="Generating summaries"):
    predictions.append(generate_summary(dialogue))

test["prediction"] = predictions

test.to_csv(
    "/content/qwen_08b_zero_shot.csv",
    index=False
)

# 4. LLM Lora

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

TRAIN_SIZE = 3000
train_data = train[["dialogue", "summary"]].iloc[:TRAIN_SIZE].copy()
validation_data = validation[["dialogue", "summary"]].copy()


def make_text(row):
    return f"""Summarize the following dialogue.
Write one concise sentence describing the main event or action.
Do not add information that is not stated in the dialogue.
Output only the summary.

Dialogue:
{row["dialogue"]}

Summary:
{row["summary"]}"""


train_data["text"] = train_data.apply(
    make_text,
    axis=1
)

validation_data["text"] = validation_data.apply(
    make_text,
    axis=1
)


train_dataset = Dataset.from_pandas(
    train_data[["text"]],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_data[["text"]],
    preserve_index=False
)


print("Train examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    trust_remote_code=True
)

print("Device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)


training_args = SFTConfig(
    output_dir="./qwen3.5-0.8b-lora",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    fp16=True,

    gradient_checkpointing=False,

    optim="adamw_torch",

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    max_length=512,

    dataset_text_field="text",

    packing=False
)


trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    peft_config=lora_config
)


print("\nStarting LoRA training...\n")

trainer.train()


trainer.save_model("./qwen3.5-0.8b-lora")

tokenizer.save_pretrained(
    "./qwen3.5-0.8b-lora"
)

print("\nTraining finished!")
print("LoRA adapter saved to:")
print("./qwen3.5-0.8b-lora")

In [ ]:
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
ADAPTER_PATH = "/content/qwen3.5-0.8b-lora"
MERGED_PATH = "/content/qwen3.5-0.8b-lora-merged"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model = model.merge_and_unload()
model.save_pretrained(
    MERGED_PATH,
    safe_serialization=True
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

tokenizer.save_pretrained(MERGED_PATH)

print(f"Готово: {MERGED_PATH}")

In [ ]:
MODEL_NAME = "/content/qwen3.5-0.8b-lora-merged"

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

def make_prompt(dialogue):
    return f"""Summarize the following dialogue.
Write one concise sentence describing the main event or action.
Do not add information that is not stated in the dialogue.
Output only the summary.

Dialogue:
{dialogue}

Summary:"""

def generate_summary(dialogue):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": make_prompt(dialogue)
            }
        ],
        temperature=0,
        max_tokens=128
    )

    return response.choices[0].message.content.strip()


predictions = []

for i, row in tqdm(
    test.iterrows(),
    total=len(test)
):
    prediction = generate_summary(row["dialogue"])
    predictions.append(prediction)

    if (i + 1) % 50 == 0:
        pd.DataFrame({
            "id": test.iloc[:i+1]["id"],
            "reference": test.iloc[:i+1]["summary"],
            "prediction": predictions
        }).to_csv(
            "/content/qwen_08b_lora.csv",
            index=False
        )


pred_df = pd.DataFrame({
    "id": test["id"],
    "reference": test["summary"],
    "prediction": predictions
})

pred_path = "/content/qwen_08b_lora.csv"

pred_df.to_csv(
    pred_path,
    index=False
)

# 5. Метрики BLEU, ROUGE - 1, ROUGE - 2, ROUGE - L

In [ ]:
def calculate_metrics(predictions, references):

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    )

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    rouge_scores = [
        scorer.score(ref, pred)
        for pred, ref in zip(predictions, references)
    ]

    rouge1 = sum(
        score["rouge1"].fmeasure
        for score in rouge_scores
    ) / len(rouge_scores)

    rouge2 = sum(
        score["rouge2"].fmeasure
        for score in rouge_scores
    ) / len(rouge_scores)

    rougeL = sum(
        score["rougeL"].fmeasure
        for score in rouge_scores
    ) / len(rouge_scores)

    return {
        "BLEU": bleu.score,
        "ROUGE-1": rouge1,
        "ROUGE-2": rouge2,
        "ROUGE-L": rougeL
    }

model_files = {
    "Baseline":
        "/content/baseline_predictions.csv",

    "Qwen 0.8B Zero-shot":
        "/content/qwen_08b_zero_shot.csv",

    "Qwen 0.8B LoRA":
        "/content/qwen_08b_lora.csv",

    "Qwen 27B Zero-shot":
        "/content/qwen_27b_zero_shot.csv",

    "Qwen 27B LoRA":
        "/content/qwen_27b_lora.csv",
}


references = test["summary"].tolist()

results = []

for model_name, path in model_files.items():

    df = pd.read_csv(path)
    predictions = df["prediction"].fillna("").tolist()

    metrics = calculate_metrics(
        predictions,
        references
    )

    results.append({
        "Model": model_name,
        **metrics
    })


results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("RESULTS")
print("=" * 75)

display(
    results_df.style.format({
        "BLEU": "{:.4f}",
        "ROUGE-1": "{:.4f}",
        "ROUGE-2": "{:.4f}",
        "ROUGE-L": "{:.4f}",
    })
)

In [ ]:
random.seed(42)

sample_indices = random.sample(range(len(test)), 10)

predictions = {}

for model_name, path in model_files.items():
    df = pd.read_csv(path)
    predictions[model_name] = df["prediction"].fillna("").tolist()


rows = []

for idx in sample_indices:

    row = {
        "Example": idx,
        "Reference": test.iloc[idx]["summary"],
    }

    for model_name, preds in predictions.items():
        row[model_name] = preds[idx]

    rows.append(row)


comparison_df = pd.DataFrame(rows)

comparison_df = comparison_df.rename(columns={
    "Qwen 0.8B Zero-shot": "0.8B Zero-shot",
    "Qwen 0.8B LoRA": "0.8B LoRA",
    "Qwen 27B Zero-shot": "27B Zero-shot",
    "Qwen 27B LoRA": "27B LoRA",
})


html = """
<style>

.model-table {
    border-collapse: collapse;
    width: 100%;
    table-layout: fixed;
    font-family: Arial, sans-serif;
    font-size: 14px;
}

.model-table th {
    background-color: #f0f0f0;
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}

.model-table td {
    border: 1px solid #ccc;
    padding: 10px;
    vertical-align: top;
    white-space: normal !important;
    overflow-wrap: break-word;
    word-wrap: break-word;
    line-height: 1.4;
}

.model-table th:nth-child(1),
.model-table td:nth-child(1) {
    width: 5%;
}

.model-table th:nth-child(2),
.model-table td:nth-child(2) {
    width: 18%;
}

.model-table th:nth-child(3),
.model-table td:nth-child(3) {
    width: 15%;
}

.model-table th:nth-child(4),
.model-table td:nth-child(4) {
    width: 15%;
}

.model-table th:nth-child(5),
.model-table td:nth-child(5) {
    width: 15%;
}

.model-table th:nth-child(6),
.model-table td:nth-child(6) {
    width: 15%;
}

.model-table th:nth-child(7),
.model-table td:nth-child(7) {
    width: 15%;
}

</style>
"""

html += comparison_df.to_html(
    index=False,
    classes="model-table",
    escape=True
)

display(HTML(html))

# 6. LLM-as-a-judge

In [33]:
baseline_predictions = pd.read_csv(
    "/content/baseline_predictions.csv"
)["prediction"].tolist()

qwen_08b_zero_predictions = pd.read_csv(
    "/content/qwen_08b_zero_shot.csv"
)["prediction"].tolist()

qwen_08b_lora_predictions = pd.read_csv(
    "/content/qwen_08b_lora.csv"
)["prediction"].tolist()

qwen_27b_zero_predictions = pd.read_csv(
    "/content/qwen_27b_zero_shot.csv"
)["prediction"].tolist()

qwen_27b_lora_predictions = pd.read_csv(
    "/content/qwen_27b_lora.csv"
)["prediction"].tolist()

In [ ]:
N_EXAMPLES = 420
SEED = 2026

DAILY_LIMIT = 500_000
JUDGE_MODEL = "allam-2-7b"

client = Groq(api_key=API_KEY)

model_predictions = {
    "Baseline": baseline_predictions,
    "Qwen 0.8B Zero-shot": qwen_08b_zero_predictions,
    "Qwen 0.8B LoRA": qwen_08b_lora_predictions,
    "Qwen 27B Zero-shot": qwen_27b_zero_predictions,
    "Qwen 27B LoRA": qwen_27b_lora_predictions,
}

model_names = list(model_predictions.keys())

rng = random.Random(SEED)

all_indices = list(range(len(test)))

selected_indices = rng.sample(
    all_indices,
    N_EXAMPLES
)


def judge_example(dialogue, reference, candidates):

    prompt = f"""Evaluate 5 summaries using the dialogue and reference.

Score each 1-5:
F = faithfulness: factual accuracy, no unsupported claims.
R = relevance: captures important information.
C = conciseness: brief and focused.

Return ONLY valid JSON in exactly this format:
{{"A":[F,R,C],"B":[F,R,C],"C":[F,R,C],"D":[F,R,C],"E":[F,R,C]}}

Each F, R, C must be an integer from 1 to 5.
Do not include any other text.

DIALOGUE:
{dialogue}

REFERENCE:
{reference}

A: {candidates["A"]}
B: {candidates["B"]}
C: {candidates["C"]}
D: {candidates["D"]}
E: {candidates["E"]}
"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a strict evaluator of dialogue summaries."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        response_format={
            "type": "json_object"
        }
    )

    result = json.loads(
        response.choices[0].message.content
    )

    for letter in ["A", "B", "C", "D", "E"]:
        assert letter in result
        assert len(result[letter]) == 3
        assert all(
            isinstance(x, int) and 1 <= x <= 5
            for x in result[letter]
        )

    usage = response.usage

    return result, {
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens
    }


results = []
usage_rows = []

total_tokens_used = 0

print("\n" + "=" * 70)
print("STARTING LLM-AS-A-JUDGE")
print("=" * 70)

for n, idx in enumerate(selected_indices, 1):

    row = test.iloc[idx]

    dialogue = row["dialogue"]
    reference = row["summary"]

    shuffled_models = model_names.copy()
    rng.shuffle(shuffled_models)

    mapping = {
        "A": shuffled_models[0],
        "B": shuffled_models[1],
        "C": shuffled_models[2],
        "D": shuffled_models[3],
        "E": shuffled_models[4]
    }

    candidates = {
        letter: model_predictions[model][idx]
        for letter, model in mapping.items()
    }

    try:

        scores, usage = judge_example(
            dialogue=dialogue,
            reference=reference,
            candidates=candidates
        )

        total_tokens_used += usage["total_tokens"]

        for letter in ["A", "B", "C", "D", "E"]:

            f_score, r_score, c_score = scores[letter]

            results.append({
                "test_index": idx,
                "id": row["id"],
                "candidate": letter,
                "model": mapping[letter],
                "Faithfulness": f_score,
                "Relevance": r_score,
                "Conciseness": c_score,
                "Total": (
                    f_score +
                    r_score +
                    c_score
                )
            })

        usage_rows.append({
            "test_index": idx,
            "id": row["id"],
            "input_tokens": usage["input_tokens"],
            "output_tokens": usage["output_tokens"],
            "total_tokens": usage["total_tokens"]
        })

        remaining = DAILY_LIMIT - total_tokens_used

        print(
            f"[{n:3d}/{N_EXAMPLES}] "
            f"idx={idx:<4} | "
            f"tokens={usage['total_tokens']:<5} | "
            f"total={total_tokens_used:,} | "
            f"remaining={remaining:,}"
        )

        if total_tokens_used >= DAILY_LIMIT:

            print("\nДостигнут дневной лимит токенов.")
            break

    except Exception as e:

        print(
            f"[{n:3d}/{N_EXAMPLES}] "
            f"idx={idx} | ERROR: {e}"
        )

        time.sleep(2)


judge_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("FINISHED")
print("=" * 70)


summary_df = (
    judge_df
    .groupby("model")[
        [
            "Faithfulness",
            "Relevance",
            "Conciseness",
            "Total"
        ]
    ]
    .mean()
    .sort_values(
        "Total",
        ascending=False
    )
)

summary_df = summary_df.round(3)


print("\n" + "=" * 70)
print("LLM-AS-A-JUDGE SUMMARY")
print("=" * 70)

display(summary_df)


output_path = "/content/llm_judge_420_summary.csv"

summary_df.to_csv(
    output_path,
    index=True
)

print(
    f"\nИтоговая таблица сохранена:"
    f"\n{output_path}"
)